# Amazon Bedrockを使用したAWSomeイベントエージェントの構築
---

## 概要

このノートブックでは、以下の機能を持つ完全な **エージェント型RAGシステム** の構築方法を説明します:
- ナレッジベースを使用してAWS re:Inventセッションに関する質問に回答
- ユーザーの好みと会話履歴を記憶
- パーソナライズされた推奨事項を提供
- 認証を使用して安全かつスケーラブルに実行

---

## Architecture 概要

<div style="text-align:left">
    <img src="architecture_images/image_1.png" width="80%"/>
</div>

---

## 章構成

1. **第1章: エージェント** - インテリジェントエージェントの作成
2. **第2章: ナレッジベース** - セッション情報の保存と取得
3. **第3章: メモリ** - パーソナライゼーションのための長期記憶の追加
4. **第4章: ランタイム** - 安全かつスケーラブルなデプロイ
5. **第5章: アイデンティティ** - Cognitoによる認証の追加

---

## 前提条件

**ソフトウェア要件:**
- Python 3.10以降
- Jupyter notebook環境または互換性のあるIDE
- pipパッケージマネージャー

**AWS要件:**
- 以下のサービスに対する適切な権限を持つAWS認証情報:
  - Amazon Bedrock (モデルアクセスとナレッジベース)
  - Amazon S3 (バケット作成とオブジェクトストレージ)
  - Amazon S3 Vectors (ベクトルバケットとインデックス作成)
  - Amazon Cognito (ユーザープール管理)
  - AWS IAM (ロールとポリシー管理)
  - AgentCore Memory (作成、読み取り、書き込み)
  - AgentCore Runtime (エージェントのデプロイ)
  - Amazon ECR (コンテナレジストリ)

**必要な知識:**
- ベクトルデータベースの基本的な理解
- AWSサービスに関する知識
- RAG (Retrieval-Augmented Generation) の概念の理解
- AIエージェントとLLMの理解
- メモリシステムに関する知識

## セットアップ: 依存関係のインストール

In [ ]:
%pip install -qUr requirements.txt

In [ ]:
!pip uninstall --yes bedrock-agentcore-starter-toolkit 
!npm install -g @aws/agentcore

In [ ]:
import os
import json
import uuid
import time
import boto3
import logging
import pandas as pd
import requests
from io import StringIO
from datetime import datetime
from utils import create_agentcore_role
from IPython.display import Markdown, display

from botocore.exceptions import ClientError

# ログ設定
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("awsome-event-agent")

# 設定
REGION = os.getenv("AWS_REGION", "us-west-2")
UNIQUE_ID = str(uuid.uuid4())[:8]
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"

print("=" * 70)
print("🚀 AWSOME EVENT AGENT - 完全ワークショップ")
print("=" * 70)
print(f"\n✅ 設定:")
print(f"   リージョン: {REGION}")
print(f"   一意ID: {UNIQUE_ID}")
print(f"   モデル: {MODEL_ID}")
print(f"   埋め込みモデル: {EMBEDDING_MODEL}")

---

# 第1章: エージェント

## 概要

この章では、 **Strands** フレームワークを使用してインテリジェントエージェントを作成します。

### 構築するもの

1. **シンプルエージェント**: 基本的な会話エージェント

### Strandsとは？

Strands は以下の機能を提供するエージェントフレームワークです:
- **モデル抽象化**: 任意のLLM (Bedrock、OpenAIなど) で動作
- **ツール統合**: エージェント向けの簡単な関数呼び出し
- **フックシステム**: カスタムロジック用のライフサイクルイベント
- **状態管理**: 永続的なエージェント状態
- **ストリーミングサポート**: リアルタイム応答ストリーミング

<div style="text-align:left">
    <img src="architecture_images/image_2.png" width="80%"/>
</div>


## ステップ 1.1: シンプルなエージェントを作成する

まず、ツールやメモリを持たない基本的な会話エージェントを作成します。

**エージェントコンポーネント:**
- **モデル**: Claude Haiku 4.5をラップしたBedrockModel
- **システムプロンプト**: エージェントの役割と動作を定義する
- **ツールなし**: エージェントはチャットのみ可能で、情報の取得はできない

**期待される動作:**
re:Inventのセッションについて質問された場合、エージェントはデータにアクセスできないため、その旨を回答します。


In [ ]:
print("\n" + "=" * 70)
print("🤖 シンプルエージェントを作成します")
print("=" * 70)

from strands import Agent
from strands.models import BedrockModel

# モデルIDを指定
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
# モデルを作成
model = BedrockModel(model_id=MODEL_ID)

# シンプルなエージェントの作成
# system_promptの引数にエージェントの概要を指示
simple_agent = Agent(
    model=model, system_prompt="あなたは非常に優秀な\
        AWS Summit 2026のセッションを知り尽くしたアシスタントです。")

# シンプルなエージェントを実行し回答を取得する
response = simple_agent("AI Agentsを学習するのに良いセッションを教えて")


print("\n✅ シンプルエージェントの作成完了")

# テスト
print("\n📝 シンプルエージェントをテスト...\n")

print("\n⚠️ 注意: エージェントはまだセッションデータにアクセスできません！")

---

# 第2章: ナレッジベース

## 概要

この章では、AWS Summit 2026のセッションデータを使って**Amazon Bedrockナレッジベース**を作成します。

### ナレッジベースとは？

Amazon Bedrockナレッジベースは、フルマネージドのRAG（検索拡張生成）ソリューションで、以下の機能を提供します：
- **自動取り込み**: ドキュメントを自動的に処理・チャンク分割
- **埋め込み生成**: 基盤モデルを使用してベクトル表現を作成
- **ベクトルDBへの保存**: ベクトルストレージとインデックスを管理
- **検索APIの提供**: セマンティック検索のためのシンプルなインターフェース
- **自動スケーリング**: インフラ管理が不要

<div style="text-align:left">
 <img src="architecture_images/image_4.png" width="80%"/>
</div>

### この章で行うこと

1. AWS Summit 2026のセッションデータをダウンロード（583セッション）
2. S3バケットを作成し、ドキュメントをアップロード
3. S3ベクトルストアとインデックスを作成
4. 自動取り込み機能付きのナレッジベースを作成
5. 検索のテスト
6. ツール付きエージェント: ナレッジベース検索機能を追加
7. RAGのテスト: エージェントがKBを使って質問に回答する様子を確認


## ステップ2.1: AWS Summit 2026 セッションデータの確認

公開ソースからAWS Summit 2026のセッションカタログを事前に用意しています。このデータセットには以下が含まれています：
- セッションコード、タイプ、タイトル、説明
- AI/MLからセキュリティ、データベースなど幅広いトピック

### データソース

データは以下を含むキュレートされたTSV（タブ区切り値）ファイルから取得されます：
- **セッションID**: 一意識別子（例：「AIM301」、「SEC401」）
- **セッションタイトル**: セッションの説明的な名称
- **セッション日時**: セッションが実施される日時
- **room**: セッションが実施される場所
- **セッション説明**: セッション内容の詳細な概要

In [ ]:
print("\n" + "=" * 70)
print("📥 AWS SUMMIT 2026 セッションデータのダウンロード")
print("=" * 70)

# CSVデータの解析
df = pd.read_csv('./data/aws_summit_japan_2026_sessions.csv')

print(f"\n✅ {len(df)}セッションをダウンロード")
print(f"\nカラム: {list(df.columns)}")
print(f"\nサンプルセッション:")
print(df.iloc[0].to_dict())
df.head()

## ステップ2.2: ナレッジベース用ドキュメントの準備

次に、生のセッションデータをベクトル検索に最適化された個別のテキストドキュメントに変換します。

### ドキュメント構造

各セッションは以下の構造を持つテキストファイルに変換されます：
```
Title: Kiro と Amazon の文化から学ぶ AI 駆動開発の型
Date: 2026-06-25 15:30-16:10
Room: Hall 8: Room 05

Description:
AI がソフトウェアを書くことが当たり前になり、開発の間口はさらに大きく広がりました。しかし、AI 駆動開発の実践には依然として人間が身につけるべきスキルが多く、人によって解釈が異なるため組織への浸透も容易ではありません。本セッションでは、Kiro の仕様駆動開発や Everything as Code を AI 駆動開発の型として取り上げます。AWS のエンジニアたちの学びと挑戦のストーリーとともに、AI 駆動開発をコードを書く技術から課題解決の手段へと変えるヒントをお届けします。2025 年版をさらに深めた内容です。

```

### なぜ個別ファイルにするのか？

セッションごとに1ファイルを作成することで：
- **きめ細かい検索**: 各セッションを独立してマッチング可能
- **明確な帰属**: どのドキュメントにどの情報があるか追跡が容易
- **柔軟な更新**: 全体を再処理せずにセッションの追加・削除が可能
- **最適なチャンキング**: Bedrockが各セッションを適切にチャンク分割可能


In [ ]:
print("\n" + "=" * 70)
print("📝 セッションドキュメントの準備")
print("=" * 70)

# ドキュメントディレクトリの作成
docs_dir = "aws_sessions_docs"
os.makedirs(docs_dir, exist_ok=True)

# セッションごとに1つのテキストファイルを作成
session_files = []
for idx, row in df.iterrows():
    session_doc = f"""Title: {row["title"]}
Date: {row["date"]}
Room: {row["room"]}

Description:
{row["detail"]}

"""

    filename = f"{row['session_id']}.txt"
    filepath = os.path.join(docs_dir, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(session_doc)

    session_files.append(filename)

    if (idx + 1) % 100 == 0:
        print(f"   {idx + 1}/{len(df)} ドキュメントを作成...")

print(f"\n✅ {len(session_files)}個のドキュメントファイルを作成")
print(f"   保存先: {docs_dir}/")

## ステップ2.3: S3バケットの作成とドキュメントのアップロード

Amazon S3バケットを作成し、準備したドキュメントをアップロードします。このバケットはBedrockナレッジベースの取り込みソースとして機能します。

### S3バケットの設定

バケットは以下の設定で作成されます：
- **リージョン固有のロケーション**: Bedrockアクセスの低レイテンシーを確保
- **標準ストレージクラス**: ナレッジベースドキュメントに対してコスト効率が良い
- **プレーンテキストコンテンツタイプ**: テキスト処理に最適化

### アップロードプロセス

全175ドキュメントをバケット内の `sessions/` プレフィックスにアップロードします。この構成により：
- ドキュメントが論理的にグループ化される
- データソース設定が容易になる
- 追加のドキュメントタイプによる将来の拡張が可能

> ⚠️ **バケット命名**: S3バケット名はグローバルに一意である必要があります。一意性を確保するためにUUIDサフィックスを使用します。


In [ ]:
print("\n" + "=" * 70)
print("📦 S3バケットを作成してドキュメントをアップロード")

print("=" * 70)

# クライアントの初期化
s3_client = boto3.client("s3", region_name=REGION)
sts_client = boto3.client("sts", region_name=REGION)
account_id = sts_client.get_caller_identity()["Account"]

BUCKET_NAME = f"bedrock-agentcore-kb-{UNIQUE_ID}"

# S3バケットの作成
print(f"\n📦 S3バケットを作成中: {BUCKET_NAME}")
try:
    if REGION == "us-east-1":
        s3_client.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3_client.create_bucket(
            Bucket=BUCKET_NAME, CreateBucketConfiguration={"LocationConstraint": REGION}
        )
    print(f"✅ バケット作成完了")
except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print(f"✅ バケットは既に存在します")

# ドキュメントのアップロード
print(f"\n📤 {len(session_files)}個のドキュメントをアップロード中...")
uploaded_count = 0
for filename in session_files:
    filepath = os.path.join(docs_dir, filename)
    s3_key = f"sessions/{filename}"

    with open(filepath, "rb") as f:
        s3_client.put_object(
            Bucket=BUCKET_NAME, Key=s3_key, Body=f, ContentType="text/plain"
        )

    uploaded_count += 1
    if uploaded_count % 100 == 0:
        print(f"   {uploaded_count}/{len(session_files)} ファイルをアップロード...")

print(f"\n✅ {uploaded_count}個のドキュメントを s3://{BUCKET_NAME}/sessions/ にアップロード完了")

## ステップ2.4: ナレッジベース用IAMロールの作成

ナレッジベースがAWSリソースにアクセスするためにIAMロールが必要です。このロールには以下の権限が付与されます：
- S3バケットからのドキュメント読み取り
- Titan Embeddingsモデルの呼び出し
- S3ベクトルインデックスの管理

### 信頼ポリシー

信頼ポリシーにより：
- `bedrock.amazonaws.com` がこのロールを引き受けることを許可
- Bedrockサービスのみがこれらの権限を使用できることを保証

### 権限ポリシー

ポリシーは以下へのアクセスを付与します：
- **S3操作**: GetObject、ListBucket
- **Bedrockモデル**: 埋め込み生成のためのInvokeModel
- **S3 Vectors**: ベクトルバケットとインデックスに対する完全なCRUD操作

> 💡 **セキュリティ**: これは最小権限の原則に従い、KB操作に必要なもののみを付与します。


In [ ]:
print("\n" + "=" * 70)
print("🔐 ナレッジベース用IAMロールの作成")
print("=" * 70)

iam_client = boto3.client("iam", region_name=REGION)
kb_role_name = f"BedrockKBRole_{UNIQUE_ID}"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    create_role_response = iam_client.create_role(
        RoleName=kb_role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Role for Bedrock Knowledge Base",
    )
    kb_role_arn = create_role_response["Role"]["Arn"]
    print(f"\n✅ ロール作成完了: {kb_role_arn}")
except iam_client.exceptions.EntityAlreadyExistsException:
    kb_role_arn = f"arn:aws:iam::{account_id}:role/{kb_role_name}"
    print(f"\n✅ ロールは既に存在します: {kb_role_arn}")

# ポリシーのアタッチ
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*",
            ],
        },
        {
            "Effect": "Allow",
            "Action": "bedrock:InvokeModel",
            "Resource": f"arn:aws:bedrock:{REGION}::foundation-model/{EMBEDDING_MODEL}",
        },
        {
            "Effect": "Allow",
            "Action": ["s3vectors:*"],
            "Resource": f"arn:aws:s3vectors:{REGION}:{account_id}:bucket/*",
        },
    ],
}

iam_client.put_role_policy(
    RoleName=kb_role_name,
    PolicyName=f"BedrockKBPolicy_{UNIQUE_ID}",
    PolicyDocument=json.dumps(policy_document),
)
print(f"✅ ポリシーをロールにアタッチ完了")

print("\n⏳ IAMロールの伝播を待機中（30秒）...")
time.sleep(30)

## ステップ2.5: S3ベクトルストアとインデックスの作成

### S3 Vectorsとは？

Amazon S3 Vectorsは、ベクトルの保存とクエリをネイティブにサポートする初のクラウドオブジェクトストアです。AIエージェント、AI推論、Amazon S3に保存されたコンテンツのセマンティック検索のために、目的に特化したコスト最適化されたベクトルストレージを提供します。クラスター管理なしで自動的にスケールし、Amazon Bedrockナレッジベースとネイティブに統合されます。

### ベクトルストア（ベクトルバケット）

S3ベクトルストア（ベクトルバケットとも呼ばれる）は、ベクトル操作に最適化された特殊なS3バケットです。AWSによるフルマネージドでインフラの保守が不要、バケットごとに複数のインデックスをサポートし、使用量に基づいて自動的にスケールします。

### ベクトルインデックスの設定

ベクトルインデックスは3つの主要な設定で埋め込みを保存・検索します：

- **次元数 (1024)**: Titan Embeddings V2の出力に一致し、モデルからの豊富なセマンティック情報をキャプチャします。

- **距離メトリック (コサイン)**: 大きさではなくベクトルの方向を比較して類似性を測定し、テキスト埋め込みに最適です。

- **データ型 (Float32)**: セマンティック検索操作における精度とストレージ効率のバランスを取ります。


In [ ]:
print("\n" + "=" * 70)
print("📊 S3ベクトルストアとインデックスの作成")
print("=" * 70)

s3vectors_client = boto3.client("s3vectors", region_name=REGION)
vector_store_name = f"bedrock-kb-{UNIQUE_ID}"
vector_index_name = f"reinvent-sessions-index-{UNIQUE_ID}"

# ベクトルバケットの作成
print(f"\n📊 S3ベクトルストアを作成中: {vector_store_name}")
try:
    s3vectors_client.create_vector_bucket(vectorBucketName=vector_store_name)
    print(f"✅ ベクトルバケット作成完了")
except Exception as e:
    print(f"✅ ベクトルバケットは既に存在または作成済み: {e}")

# ベクトルバケットARNの取得
bucket_response = s3vectors_client.get_vector_bucket(vectorBucketName=vector_store_name)
vector_store_arn = bucket_response["vectorBucket"]["vectorBucketArn"]
print(f"   ARN: {vector_store_arn}")

# ベクトルインデックスの作成
print(f"\n📊 ベクトルインデックスを作成中: {vector_index_name}")
try:
    s3vectors_client.create_index(
        vectorBucketName=vector_store_name,
        indexName=vector_index_name,
        dimension=1024,  # Titan Embeddings V2
        distanceMetric="cosine",
        dataType="float32",
        metadataConfiguration={"nonFilterableMetadataKeys": ["AMAZON_BEDROCK_TEXT"]},
    )
    print(f"✅ ベクトルインデックス作成完了")
except Exception as e:
    print(f"✅ ベクトルインデックスは既に存在または作成済み: {e}")

# インデックスARNの取得
index_response = s3vectors_client.list_indexes(vectorBucketName=vector_store_name)
vector_index_arn = index_response["indexes"][0]["indexArn"]
print(f"   ARN: {vector_index_arn}")

## ステップ2.6: ナレッジベースの作成とデータ取り込み

### ナレッジベースのオーケストレーション

<div style="text-align:left">
    <img src="architecture_images/image_3.png" width="80%"/>
</div>

次に、RAGパイプライン全体をオーケストレーションするBedrockナレッジベースを作成します。

**ナレッジベースのフロー**:
1. **ドキュメント取り込み**: Amazon S3からドキュメントを読み取り
2. **テキスト抽出**: ドキュメントコンテンツを解析
3. **チャンキング**: ドキュメントを最適なセグメントに分割（300トークン、20%オーバーラップ）
4. **埋め込み生成**: Titan V2を使用してベクトルを生成
5. **インデックス作成**: S3 Vectorsにベクトルを保存

**設定の詳細**:
- **タイプ**: VECTOR（セマンティック検索用）
- **埋め込みモデル**: Titan Embeddings V2
- **ストレージ**: S3 Vectors（サーバーレス）
- **次元数**: 1024
- **データ型**: FLOAT32

### チャンキング戦略

**固定サイズチャンキング**:
- **最大トークン数**: チャンクあたり300
- **オーバーラップ**: 連続するチャンク間で20%
```


In [ ]:
print("\n" + "=" * 70)
print("📚 ナレッジベースの作成")
print("=" * 70)

bedrock_agent_client = boto3.client("bedrock-agent", region_name=REGION)
KB_NAME = f"AWSSummitSessionsKB_{UNIQUE_ID}"

# ナレッジベースの作成
print(f"\n📚 ナレッジベースを作成中: {KB_NAME}")
kb_response = bedrock_agent_client.create_knowledge_base(
    name=KB_NAME,
    description="AWS Summit 2026 sessions knowledge base",
    roleArn=kb_role_arn,
    knowledgeBaseConfiguration={
        "type": "VECTOR",
        "vectorKnowledgeBaseConfiguration": {
            "embeddingModelArn": f"arn:aws:bedrock:{REGION}::foundation-model/{EMBEDDING_MODEL}",
            "embeddingModelConfiguration": {
                "bedrockEmbeddingModelConfiguration": {
                    "dimensions": 1024,
                    "embeddingDataType": "FLOAT32",
                }
            },
        },
    },
    storageConfiguration={
        "type": "S3_VECTORS",
        "s3VectorsConfiguration": {"indexArn": vector_index_arn},
    },
)

KB_ID = kb_response["knowledgeBase"]["knowledgeBaseId"]
print(f"✅ ナレッジベース作成完了: {KB_ID}")

# KBの準備完了を待機
print("\n⏳ ナレッジベースがACTIVEになるのを待機中...")
while True:
    kb_status = bedrock_agent_client.get_knowledge_base(knowledgeBaseId=KB_ID)
    status = kb_status["knowledgeBase"]["status"]
    if status == "ACTIVE":
        print(f"✅ ナレッジベースがACTIVEになりました")
        break
    elif status == "FAILED":
        print(f"❌ ナレッジベースの作成に失敗しました")
        break
    print(f"   Status: {status}", end="\r")
    time.sleep(5)

### ステップ2.6.1: データソースの作成と取り込み開始

ナレッジベースセットアップの最終ステップは、データソースの作成とドキュメントの取り込みです。

**データソースの設定**:
- **タイプ**: S3
- **バケット**: 先ほど作成したバケット
- **チャンキング**: 20%オーバーラップの固定サイズ
- **自動処理**: Bedrockが解析と埋め込みを処理

**取り込みプロセス**:
1. S3バケットをスキャン（583ファイル）
2. 各テキストファイルを解析
3. 300トークンのセグメントにチャンク分割
4. 埋め込みを生成（Titan V2）
5. S3 Vectorsインデックスに保存


In [ ]:
print("\n" + "=" * 70)
print("📁 データソースの作成とデータ取り込み")
print("=" * 70)

# データソースの作成
print(f"\n📁 データソースを作成中...")
ds_response = bedrock_agent_client.create_data_source(
    knowledgeBaseId=KB_ID,
    name=f"ReInventSessions_DataSource_{UNIQUE_ID}",
    description="S3 data source with re:Invent session documents",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {"bucketArn": f"arn:aws:s3:::{BUCKET_NAME}"},
    },
    vectorIngestionConfiguration={
        "chunkingConfiguration": {
            "chunkingStrategy": "FIXED_SIZE",
            "fixedSizeChunkingConfiguration": {
                "maxTokens": 300,
                "overlapPercentage": 20,
            },
        }
    },
)

data_source_id = ds_response["dataSource"]["dataSourceId"]
print(f"✅ データソース作成完了: {data_source_id}")

# 取り込み開始
print("\n🔄 データ取り込みを開始...")
ingestion_response = bedrock_agent_client.start_ingestion_job(
    knowledgeBaseId=KB_ID, dataSourceId=data_source_id
)

ingestion_job_id = ingestion_response["ingestionJob"]["ingestionJobId"]
print(f"   取り込みジョブ開始: {ingestion_job_id}")

# 取り込み完了を待機
print("⏳ 取り込み完了を待機中（数分かかる場合があります）...")
while True:
    job_status = bedrock_agent_client.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=data_source_id,
        ingestionJobId=ingestion_job_id,
    )
    status = job_status["ingestionJob"]["status"]
    if status == "COMPLETE":
        stats = job_status["ingestionJob"].get("statistics", {})
        print(f"\n✅ 取り込み完了！")
        print(f"   処理済みドキュメント数: {stats.get('numberOfDocumentsScanned', 0)}")
        break
    elif status == "FAILED":
        failures = job_status["ingestionJob"].get("failureReasons", [])
        print(f"\n❌ 取り込み失敗: {failures}")
        break
    print(f"   Status: {status}", end="\r")
    time.sleep(15)

## ステップ2.7: ナレッジベース取得のテスト

ナレッジベースの検索が正しく動作していることをテストしましょう。

### 検索API

`retrieve` APIは：
- **クエリを埋め込み**: 同じモデル（Titan V2）を使用してテキストをベクトルに変換
- **ベクトルを検索**: コサイン類似度を使用して最も類似したチャンクを検索
- **結果を返却**: 関連性スコア付きのTop Kチャンク
- **メタデータを含む**: ソースドキュメントと位置情報


In [ ]:
print("\n" + "=" * 70)
print("🧪 ナレッジベース検索のテスト")
print("=" * 70)

bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)

# テストクエリ
test_query = "Generative AI と securityについてを学習できるセッションは？"
print(f"\n🔍 Query: {test_query}")

# retrieve API を用いて予め作成したナレッジベースを検索
# セマンティック検索でチャンクを検索
response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={"text": test_query},
    retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
)

print(f"\n✅ {len(response['retrievalResults'])}件の結果が見つかりました:\n")

# 取得したレスポンスを表示用にデータ整形
for idx, result in enumerate(response["retrievalResults"], 1):
    content = result["content"]["text"]
    score = result["score"]
    print(f"{idx}. Score: {score:.3f}")
    print(f"   {content[:200]}...\n")

## ステップ2.8: ナレッジベース検索ツールの作成

次に、エージェントがナレッジベースを検索できるStrandsツールを作成します。

### Strandsツールとは？

ツールはエージェントが呼び出せるPython関数です。`@tool`デコレータは、パラメータ検証のためのPython型ヒントと自然言語記述のためのdocstringを使用して、OpenAPI互換のスキーマを生成します。エージェントは会話のコンテキストに基づいて、いつツールを呼び出すかを自動的に判断します。

### ツール設計

`search_summit_session`ツールはクエリ文字列とオプションのmax_resultsパラメータ（デフォルト5）を受け取ります。Bedrockナレッジベースの検索APIを呼び出し、類似度スコア付きの関連チャンクを抽出し、エージェントが使用するためのランク、コンテンツ、スコアを含む辞書のリストとしてフォーマットします。

エージェントは、ユーザーがre:Inventセッションについて質問したり、推奨を求めたり、セッションキーワードに関連する特定の技術情報が必要な場合に、このツールを自動的に呼び出します。



In [ ]:
print("\n" + "=" * 70)
print("🔧 ナレッジベース検索ツールの作成")
print("=" * 70)

# strands の tool をインポート
from strands import tool
from typing import List, Dict, Any

# @tool デコレータをつけた関数はエージェントによってツールとして認識される
@tool
def search_summit_session(query: str, max_results: int = 5) -> List[Dict[str, Any]]:
    # エージェントは docustring を読んで、ツールを実行するかどうかの判断を行う
    """
    セマンティック検索を使用して、ナレッジベースからAWS Summitのセッションを検索します。

    ユーザーがAWS Summitのセッションに関する推奨情報を求めた場合や、
    ユーザーの興味に基づいて関連するセッションを見つけたい場合に、このツールをご利用ください。

    Args:
        query: どのようなセッションを検索するかを指定する検索クエリ
        max_results: 返却するセッションの最大結果数 (default: 5, max: 10)

    Returns:
        関連するAWS Summit セッションの詳細が書かれたリスト
    """
    try:
        logger.info(f"🔍 Searching KB with query: {query}")

        # ナレッジベース検索APIの呼び出し
        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={"text": query},
            retrievalConfiguration={
                "vectorSearchConfiguration": {"numberOfResults": min(max_results, 10)}
            },
        )

        # 結果の抽出とフォーマット
        results = []
        for idx, item in enumerate(response.get("retrievalResults", []), 1):
            result = {
                "rank": idx,
                "content": item.get("content", {}).get("text", ""),
                "score": item.get("score", 0.0),
            }
            results.append(result)

        logger.info(f"✅ Found {len(results)} re:Invent sessions")
        return results

    except Exception as e:
        logger.error(f"❌ Error searching KB: {e}")
        return [{"error": f"Failed to search knowledge base: {str(e)}"}]


print("\n✅ ナレッジベース検索ツール作成完了")

---

# 第3章: メモリ

## 概要

この章では、AgentCoreメモリを使用してエージェントに**短期記憶**と**長期記憶**を追加します。

### AgentCoreメモリとは？

AgentCoreメモリは、セッション間でエージェントの記憶を永続的に保存するフルマネージドサービスです。

**主な機能**:
- **マルチ戦略**: UserPreference、Semantic、Episodicメモリをサポート
- **アクタースコープ**: ユーザー（アクター）ごとにメモリを分離
- **自動処理**: バックグラウンドでの分析と抽出
- **名前空間による整理**: メモリタイプの論理的な分離
- **セッション横断**: 会話履歴を超えて永続化

### メモリアーキテクチャ

<div style="text-align:left">
    <img src="architecture_images/image_5.png" width="80%"/>
</div>

### メモリ戦略

**UserPreferenceMemoryStrategy**（今回使用するもの）:
- **目的**: 会話からユーザーの好みを自動的に学習
- **ストレージ**: `/users/{actorId}/preferences` 名前空間
- **コンテンツ**: コミュニケーションスタイル、興味、背景、好み
- **処理**: LLMベースの会話コンテキスト分析
- **自動**: 明示的なコマンド不要

**動作の仕組み**:
```
ユーザー: 「私はPython開発者でAIに興味があります」
  ↓
エージェントが自然に応答
  ↓
UserPreference戦略（バックグラウンド）:
  - 会話を分析
  - 抽出: 「ユーザーはPython開発者」
  - 抽出: 「ユーザーはAIに興味がある」
  - /users/{actorId}/preferences に保存
  ↓
次の会話:
  - エージェントが好みを読み込み
  - 応答をパーソナライズ
  - Python/AIコンテンツを推奨
```

> 💡 **重要なポイント**: UserPreferenceの学習は明示的な「覚えて」コマンドなしに自動的に行われ、インタラクションが自然に感じられます。


## ステップ3.1: AgentCoreメモリの作成

UserPreference戦略を使用してメモリリソースを作成しましょう。

### メモリの設定

**名前**: `EventAgentMemory_{UNIQUE_ID}`
- このメモリリソースの一意識別子
- アカウントごとに複数のメモリリソースが可能

**戦略**: UserPreference
- 後から追加の戦略を追加可能（Semantic、Episodicなど）
- 各戦略は独自の名前空間を持つ

**名前空間**: `/users/{actorId}/preferences`
- `{actorId}` はユーザーIDで動的に置換
- ユーザーデータの分離を保証
- マルチテナントアプリケーションを実現

**イベント有効期限**: 30日
- 生の会話イベントを保持する期間
- 抽出された好みはより長く永続化
- データ保持要件に基づいて調整

> ⏱️ **作成時間**: AWSがインフラをプロビジョニングするため、メモリリソースの作成には2〜3分かかります。


In [ ]:
print("\n" + "=" * 70)
print("🧠 AgentCore Memory の作成")
print("=" * 70)

from bedrock_agentcore.memory.constants import StrategyType
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore.memory import MemoryClient


# メモリクライアントの初期化
client = MemoryClient(region_name=REGION)

print("\n🧠 UserPreference戦略でメモリを作成中...")
print("   2〜3分かかります...\n")

# User Preference戦略を持つ長期メモリを作成（作成完了まで待機）
memory = client.create_memory_and_wait(
    name=f"MyAgentMemory_{UNIQUE_ID}",
    strategies=[{
        "userPreferenceMemoryStrategy": {
            "name": "UserPreferenceExtractor",
            # namespaces は path のような構造を取る
            # これにより長期的な関心を分離できる
            # 長期メモリをどの程度細かく、もしくはグローバルに持たせるのかに基づき
            # 多くの変数を含む長いパスを定義することができる
            "namespaces": ["/users/{actorId}/preferences/"]
        }
    }],
    description="Memory for Event Agent with user preferences",
    event_expiry_days=30,
)

# 作成した memory ID を取得
# 今後このノートブックではこの長期メモリを利用する
MEMORY_ID = memory.get("id")
print(f"\n✅ メモリ作成完了！")
print(f"   メモリID: {MEMORY_ID}")
print(f"   ステータス: {memory.get('status')}")
print(f"   名前空間: /users/{{actorId}}/preferences")

## ステップ3.2: メモリフックプロバイダーの作成

### Strandsフックの理解

<div style="text-align:left">
    <img src="architecture_images/image_6.png" width="80%"/>
</div>

フックはStrandsフレームワークのライフサイクルイベントで、カスタムロジックを注入できます：

**利用可能なフックイベント:**
- **AgentInitializedEvent**: エージェント開始時（コンテキストの読み込み）
- **MessageAddedEvent**: メッセージ追加時（メモリへの保存）
- **ToolExecutedEvent**: ツール実行後（結果のログ記録）
- **RequestEnd**: サイクル完了時

<div style="text-align:left">
    <img src="architecture_images/image_7.png" width="80%"/>
</div>


In [ ]:
print("\n" + "=" * 70)
print("🔗 メモリフックプロバイダーの作成")
print("=" * 70)

from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from strands.hooks import (
    HookProvider,
    HookRegistry,
    AgentInitializedEvent,
    MessageAddedEvent,
)

# Strands Agents が提供する HookProvider クラスを継承し
# MemoryHookProvider クラスを作成
class MemoryHookProvider(HookProvider):
    """AgentCore Memory と統合するためのカスタムフックプロバイダークラス"""

    def __init__(self):
        logger.info(f"Initializing MemoryHookProvider")

        self.memory_session_manager = MemorySessionManager(MEMORY_ID, REGION)

    # エージェント初期化時に長期メモリから User Preferences を検索
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """エージェント初期化時に長期メモリから User Preferences を検索"""
        logger.info("🎯 Agent initialization - loading preferences")

        # actor_id を取得
        actor_id = event.agent.state.get("actor_id")

        if not actor_id:
            logger.warning("Missing actor_id")
            return

        try:
            # 長期メモリからユーザーの好みを取得
            preferences = self.memory_session_manager.search_long_term_memories(
                namespace_prefix=f"/users/{actor_id}/preferences",
                # 「ユーザーの設定・興味・背景」を問い合わせるクエリをハードコード
                query="ユーザーの好み、興味、経歴はどのようなものですか？",
                # 類似情報を最大 5 件取得してメッセージに追加
                top_k=5,
            )

            # User Preferences が取得できた場合
            if preferences:
                logger.info(f"✅ Loaded {len(preferences)} preferences")
                pref_messages = []

                # 取得した User Preferences のデータを整形
                for pref in preferences:
                    pref_text = pref.get("content", {}).get("text", "")
                    if pref_text:
                        try:
                            pref_json = json.loads(pref_text)        
                            pref_messages.append(
                                f"- {pref_json.get('preference', pref_text)}"
                            )
                        except:
                            pref_messages.append(f"- {pref_text}")

                if pref_messages:
                    context = "\n".join(pref_messages)
                    
                    # User Preferences をメッセージに追加
                    # これによりエージェントはユーザーの好みに関するコンテキストを
                    # 持った状態で会話を開始できる
                    event.agent.system_prompt += f" **User Preferences:** {context}"
                    logger.info("✅ Added preferences to system prompt")
            else:
                logger.info("ℹ️ No preferences found yet")

        except Exception as e:
            logger.error(f"❌ Error loading preferences: {e}")

    # 新しいメッセージが追加される度に発火する
    def on_message_added(self, event: MessageAddedEvent):
        """メモリ内にメッセージを追加"""
        logger.info("💬 Message added - storing in memory")

        # actor_idを取得
        actor_id = event.agent.state.get("actor_id")
        # session_idを取得
        session_id = event.agent.state.get("session_id")

        if not all([actor_id, session_id]):
            logger.warning("Missing required state values")
            return

        try:
            messages = event.agent.messages
            # 最後のメッセージからコンテンツとロールを取得し、
            # AgentCore Memory のセッションマネージャーへ会話メッセージとして送信
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            if last_message["role"] == "user":
                message_role = MessageRole.USER
            elif last_message["role"] == "assistant":
                message_role = MessageRole.ASSISTANT

            # actor_id と session_id を指定して
            # 短期メモリにメッセージを送信
            # これにより論理的なデータ分離が実現
            self.memory_session_manager.add_turns(
                actor_id=actor_id,
                session_id=session_id,
                messages=[ConversationalMessage(message_content, message_role)],
            )
            logger.info("✅ Message stored")

        except Exception as e:
            logger.error(f"❌ Error storing message: {e}")

    # 作成した hooks を hook registory に登録
    # これによりエージェントは初期化時やメッセージ追加時に実行すべきコールバックを認識
    def register_hooks(self, registry: HookRegistry):
        """エージェントを hook に登録"""
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)


print("\n✅ MemoryHookProviderクラス定義完了")

## ステップ3.3: メモリとRAGを持つエージェントの作成

次に、メモリフックとRAGツールを組み合わせたエージェントを作成します。

### エージェントの設定

**コンポーネント**:
- **モデル**: BedrockModel経由のClaude Haiku 4.5
- **フック**: 自動メモリ管理のためのMemoryHookProvider
- **ツール**: ナレッジベースアクセスのためのsearch_summit_session
- **システムプロンプト**: 動作の指示
- **状態**: メモリスコーピングのためのactor_idとsession_id

### 状態管理

エージェントの状態には以下が含まれます：
- **actor_id**: ユーザー識別子（後でCognitoから取得）
- **session_id**: 会話識別子（後でRuntimeから取得）
- フックが正しいメモリを読み込み/保存するために使用

### 期待される動作

このエージェントが実行されると：
1. **初期化**: 既存の好みを読み込み
2. **ユーザーメッセージ**: メモリに保存
3. **ツール使用**: 必要に応じてKBを検索
4. **エージェント応答**: 履歴としてメモリに保存
5. **バックグラウンド**: UserPreferenceが学習のために分析

> 💡 **初回実行**: 最初の会話では好みは空です。このインタラクションから学習されます。


In [ ]:
print("\n" + "=" * 70)
print("🤖 メモリとRAGを持つエージェントの作成")
print("=" * 70)

# テストIDの生成
ACTOR_ID = f"user-{UNIQUE_ID}"
session_id_1 = f"session-1-{str(uuid.uuid4())[:8]}"

print(f"\n📋 設定:")
print(f"   アクターID: {ACTOR_ID}")
print(f"   セッションID: {session_id_1}")
print(f"   メモリID: {MEMORY_ID}")

# メモリフックの初期化
memory_hook = MemoryHookProvider()

# メモリとRAGを持つエージェントの作成
memory_rag_agent = Agent(
    model=model,
    # hook を指定
    hooks=[memory_hook],
    # tool を指定 
    tools=[search_summit_session],
    system_prompt="""あなたは有能なイベントアシスタントです。手持ちのツールを使ってセッション情報を取得できます。""",
    # actor_idとsession_idを指定して実行
    state={"actor_id": ACTOR_ID, "session_id": session_id_1},
)

print("\n✅ メモリ対応RAGエージェント作成完了")

## ステップ3.4: メモリのテスト - セッション1

個人情報を共有してメモリシステムをテストしましょう。


In [ ]:
print("=" * 70)
print("🧪 テスト1: バックグラウンド学習（UserPreferenceメモリ）")
print("=" * 70)
print("\n📝 個人情報を共有中...")
print("   期待動作: エージェントが自然に応答し、バックグラウンドで学習が行われる\n")

# ユーザーの好みをプロンプトに含めてエージェントを実行
response = memory_rag_agent(
    """
    こんにちは！自己紹介をさせてください：
        - 私の名前は辻浩季です。
        - AWSでソリューションアーキテクトとして働いています
        - 好きなプログラミング言語はPythonです
        - クラウドアーキテクチャ、AI、セキュリティに興味があります
        - 概要を説明するセッションよりも、実践的な技術セッションの方が好きです
    """
)

print("\n✅ メッセージが短期記憶に保存されました")
print("⏳ UserPreference戦略がバックグラウンドで分析します...")

## ステップ3.5: 長期記憶抽出の確認

UserPreference戦略が会話を処理し、どのような好みが抽出されたかを確認しましょう。

### バックグラウンド処理

UserPreference戦略による長期記憶は：
1. **監視**: 新しい会話ターンを監視
2. **分析**: LLMを使用して好みを抽出
3. **構造化**: カテゴリ付きのJSONとしてフォーマット
4. **保存**: 長期記憶の名前空間に保存
5. **インデックス作成**: 将来の検索のために検索可能にする

### 検証方法

`list_long_term_memory_records` APIを使用して：
- ユーザーの好み名前空間をクエリ
- 保存されたすべての好みを取得
- 抽出されたJSON構造を表示

> ⏱️ **待機時間**: 処理が完了することを確認するために40秒待機します。


In [ ]:
print("\n⏳ UserPreference戦略の処理を待機中（40秒）...")
time.sleep(40)

# 抽出された User Preferences を確認
print("\n🔍 抽出された好みを確認中...")
memory_session_manager = MemorySessionManager(MEMORY_ID, REGION)
try:
    # 長期メモリに保存されている User Preferences を取得 
    response = memory_session_manager.list_long_term_memory_records(
        namespace_prefix=f"/users/{ACTOR_ID}/preferences", max_results=10
    )

    if response:
        print(f"\n✅ {len(response)}件の好みが見つかりました！\n")
        for idx, pref in enumerate(response, 1):
            content = pref.get("content", {}).get("text", "")
            print(f"{idx}. {content}…")
    else:
        print("ℹ️ まだ好みが抽出されていません（もう少し時間が必要かもしれません）")

except Exception as e:
    print(f"⚠️ 好みを取得できませんでした: {e}")

## ステップ3.6: セッション間メモリのテスト

次に重要なテストです：完全に新しいセッションでエージェントは好みを覚えているでしょうか？

### このテストが重要な理由

**新しいセッション = チャット履歴なし**:
- 異なるsession_id
- メッセージ配列なし
- 会話はクリーンな状態
- ただし同じactor_id（同じユーザー）

**メモリの永続性**:
- 好みはLTM（チャット履歴ではなく）に保存
- `on_agent_initialized`フック経由で読み込み
- システムプロンプトに注入
- エージェントは明示的な履歴なしに「覚えている」


In [ ]:
from strands.hooks import HookProvider, HookRegistry, BeforeToolCallEvent

class ToolCallLogger(HookProvider):
    """エージェントがRAGに投げたクエリを記録するフック"""
    def __init__(self):
        self.tool_calls = []

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self._on_tool_use)

    def _on_tool_use(self, event: BeforeToolCallEvent) -> None:
        self.tool_calls.append({
            "name": event.tool_use.get("name"),
            "input": event.tool_use.get("input"),
        })                                                                                                                                                                     


In [ ]:
                                                                                                                                                                                                                                                                                                                                                          
print("\n" + "=" * 70)                                                                                                                                                          
print("🧪 セッション2: 新しいセッション、同じユーザー")                                                                                                                         
print("=" * 70)                                                                                                                                                                 
                                                                                                                                                                                
# 先程とは異なる新しいセッションの作成                                                                                                                                          
session_id_2 = f"session-2-{str(uuid.uuid4())[:8]}"                                                                                                                             
print(f"\n🔄 新しいセッションIDで新しいエージェントを作成中: {session_id_2}\n")                                                                                                 
                                                                                                                                                                                
# ツール呼び出しロガーの作成                                                                                                                                                    
tool_logger = ToolCallLogger()                                                                                                                                                  
                                                                                                                                                                                
# 完全に新しいエージェントインスタンスの作成                                                                                                                                    
new_memory_hook = MemoryHookProvider()                                                                                                                                          
new_memory_rag_agent = Agent(                                                                                                                                                   
    model=model,                                                                                                                                                                
    hooks=[new_memory_hook, tool_logger],                                                                                                                                       
    tools=[search_summit_session],                                                                                                                                              
    system_prompt="""あなたは賢いアシスタントです。手持ちのツールを使ってセッション情報を取得できます。セッションを推奨する際は、上位3件を表示してください。                    
    """,                                                                                                                                                                        
    state={                                                                                                                                                                     
        "actor_id": ACTOR_ID,                                                                                                                                                   
        "session_id": session_id_2,                                                                                                                                             
    },                                                                                                                                                                          
)                                                                                                                                                                               
                                                                                                                                                                                
print("✅ 新しいエージェント作成完了（LTMから好みを読み込みます）\n")                                                                                                           
                                                                                                                                                                                
# 前のセッションの情報について質問                                                                                                                                              
print("📝 前のセッションの情報について質問中...\n")                                                                                                                             
                                                                                                                                                                                
response = new_memory_rag_agent("私はAWS Summit 2026のどのセッションに出席するのが良いですか?")                                                                                 
                                                                                                                                                                                
display(Markdown(f"**Agent:** {response.message['content'][0]['text']}"))                                                                                                       
                                                                                                                                                                                
# エージェントが自律的に生成したRAGクエリを表示                                                                                                                                 
print("\n" + "-" * 70)                                                                                                                                                          
print("🔍 エージェントが自律的に生成したRAGクエリ:")                                                                                                                            
print("-" * 70)                                                                                                                                                                 
for i, call in enumerate(tool_logger.tool_calls, 1):                                                                                                                            
    print(f"  {i}. [{call['name']}] query: \"{call['input'].get('query', call['input'])}\"")                                                                                    
                                                                                                                                                                                
print("\n✅ エージェントが長期記憶から好みを取得しました！")                                                                                                                    
print("   これはチャット履歴からではなく、UserPreference戦略からのものです")                                                                                                    
                                                                                                                                                                                
print("\n" + "=" * 70)                                                                                                                                                          
print("✅ 第3章完了: メモリが動作しています！")                                                                                                                                 
print("=" * 70)                  

---

# 第4章: ランタイム

## 概要

この章では、エージェントを**AgentCoreランタイム**にデプロイし、安全でスケーラブルな実行を実現します。

<div style="text-align:left">
    <img src="architecture_images/image_8.png" width="80%"/>
</div>

### なぜAgentCoreランタイムなのか？

**本番環境のメリット**:
- **分離**: 各セッションが分離されたmicroVMで実行
- **スケーラビリティ**: 負荷に基づく自動スケーリング
- **セキュリティ**: 制御された実行環境
- **本番対応**: エンタープライズワークロード向けに構築
- **マネージドインフラ**: サーバー管理不要
- **組み込み監視**: CloudWatchログとX-Rayトレース

### デプロイプロセス

<div style="text-align:left">
    <img src="architecture_images/image_9.png" width="80%"/>
</div>

**実行するステップ**:
1. デプロイ用のエージェントPythonファイルを作成
2. 包括的な権限を持つIAM実行ロールを作成
3. ランタイムデプロイ設定を構成
4. クラウドに起動（CodeBuildがコンテナを作成）
5. デプロイされたエージェントをテスト


## ステップ4.1: ランタイム用エージェントファイルの作成

AgentCoreランタイム環境で実行できるスタンドアロンのPythonファイルを作成する必要があります。

### ローカルエージェントとの主な違い

**ランタイム要件**:
- スタンドアロンのPythonファイルである必要がある
- `@app.entrypoint`デコレータを使用
- payloadとcontextオブジェクトを受け取る
- 文字列レスポンスを返す
- 環境変数を処理する

### ランタイムコンテキストオブジェクト

`context`パラメータは以下を提供します：
- **session_id**: 会話の一意識別子
- **request_headers**: HTTPヘッダー（認証用）

### 環境変数

エージェントには以下の注入が必要です：
- `MODEL_ID`: 使用するLLMモデル
- `MEMORY_ID`: メモリリソース識別子
- `KB_ID`: ナレッジベース識別子
- `AWS_REGION`: AWSリージョン


In [ ]:
%%writefile event_agent.py
import os
import json
import boto3
import logging
from strands import Agent, tool
from typing import Dict, Any, List
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.memory.session import MemorySessionManager
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from bedrock_agentcore.memory.constants import StrategyType, ConversationalMessage, MessageRole

# logging の設定
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger("event-agent")

# AgentCore App を初期化
app = BedrockAgentCoreApp()

# 各種環境変数を設定
MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
REGION = os.getenv('AWS_REGION')
KB_ID = os.getenv('KB_ID')

agent = None
bedrock_agent_runtime = None

# Strands Agents が提供する HookProvider クラスを継承し
# MemoryHookProvider クラスを作成
class MemoryHookProvider(HookProvider):
    """AgentCore Memory と統合するためのカスタムフックプロバイダークラス"""
    
    def __init__(self):
        logger.info(f"Initializing MemoryHookProvider")
        self.memory_session_manager = MemorySessionManager(MEMORY_ID, REGION)
    
    # エージェント初期化時に長期メモリから User Preferences を検索
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load user preferences when agent starts"""
        logger.info("🎯 Agent initialization - loading preferences")

        # actor_id を取得        
        actor_id = event.agent.state.get("actor_id")
        
        if not actor_id:
            logger.warning("Missing actor_id")
            return
        
        try:
            # 長期メモリからユーザーの好みを取得
            preferences = self.memory_session_manager.search_long_term_memories(
                namespace_prefix=f"/users/{actor_id}/preferences",
                # 「ユーザーの設定・興味・背景」を問い合わせるクエリをハードコード
                query="What are the user's preferences, interests, and background?",
                # 類似情報を最大 5 件取得してメッセージに追加
                top_k=5
            )
            
            # User Preferences が取得できた場合
            if preferences:
                logger.info(f"✅ Loaded {len(preferences)} preferences")
                pref_messages = []
                # 取得した User Preferences のデータを整形
                for pref in preferences:
                    pref_text = pref.get('content', {}).get('text', '')
                    if pref_text:
                        try:
                            pref_json = json.loads(pref_text)
                            pref_messages.append(f"- {pref_json.get('preference', pref_text)}")
                        except:
                            pref_messages.append(f"- {pref_text}")
                
                if pref_messages:
                    context = "\n".join(pref_messages)

                    # User Preferences をメッセージに追加
                    # これによりエージェントはユーザーの好みに関するコンテキストを
                    # 持った状態で会話を開始できる
                    event.agent.system_prompt += f"\n\n**User Preferences:**\n{context}"
                    logger.info("✅ Added preferences to system prompt")
            else:
                logger.info("ℹ️ No preferences found yet")
                
        except Exception as e:
            logger.error(f"❌ Error loading preferences: {e}")
    
    # 新しいメッセージが追加される度に発火する
    def on_message_added(self, event: MessageAddedEvent):
        """Store messages in memory"""
        logger.info("💬 Message added - storing in memory")
        
        # actor_idを取得
        actor_id = event.agent.state.get("actor_id")
        # session_idを取得
        session_id = event.agent.state.get("session_id")
        
        if not all([actor_id, session_id]):
            logger.warning("Missing required state values")
            return
        
        try:
            messages = event.agent.messages
            # 最後のメッセージからコンテンツとロールを取得し、
            # AgentCore Memory のセッションマネージャーへ会話メッセージとして送信
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            if last_message["role"] == "user":
                message_role = MessageRole.USER
            elif last_message["role"] == "assistant":
                message_role = MessageRole.ASSISTANT
            
            # actor_id と session_id を指定して
            # 短期メモリにメッセージを送信
            # これにより論理的なデータ分離が実現
            self.memory_session_manager.add_turns(
                actor_id=actor_id,
                session_id=session_id,
                messages=[ConversationalMessage(message_content, message_role)]
            )
            logger.info("✅ Message stored")
            
        except Exception as e:
            logger.error(f"❌ Error storing message: {e}")

    # 作成した hooks を hook registory に登録
    # これによりエージェントは初期化時やメッセージ追加時に実行すべきコールバックを認識    
    def register_hooks(self, registry: HookRegistry):
        """Register hooks with the agent"""
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

# @tool デコレータをつけた関数はエージェントによってツールとして認識される
@tool
def search_summit_session(query: str) -> List[Dict[str, Any]]:
    # エージェントは docustring を読んで、ツールを実行するかどうかの判断を行う
    """
    セマンティック検索を使用して、ナレッジベースからAWS Summitのセッションを検索します。

    ユーザーがAWS Summitのセッションに関する推奨情報を求めた場合や、
    ユーザーの興味に基づいて関連するセッションを見つけたい場合に、このツールをご利用ください。

    Args:
        query: どのようなセッションを検索するかを指定する検索クエリ
        max_results: 返却するセッションの最大結果数 (default: 5, max: 10)

    Returns:
        関連するAWS Summit セッションの詳細が書かれたリスト
    """
    global bedrock_agent_runtime, KB_ID
    
    try:
        if not bedrock_agent_runtime:
            bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name=REGION)
        
        # ナレッジベース検索APIの呼び出し
        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={'text': query},
            retrievalConfiguration={
                'vectorSearchConfiguration': {
                    'numberOfResults': 5
                }
            }
        )
        
        # 結果の抽出とフォーマット
        results = []
        for idx, item in enumerate(response.get('retrievalResults', []), 1):
            result = {
                'rank': idx,
                'content': item.get('content', {}).get('text', ''),
                'score': item.get('score', 0.0)
            }
            results.append(result)
        
        return results
        
    except Exception as e:
        logger.error(f"Error searching KB: {e}")
        return [{"error": str(e)}]

def initialize_agent(actor_id, session_id):
    """エージェントの初期化"""
    global agent
    
    # Bedrock モデルの初期化
    model = BedrockModel(model_id=MODEL_ID)
    # MemoryHookProvider の初期化
    memory_hook = MemoryHookProvider()
    
    agent = Agent(
        model=model,
        hooks=[memory_hook],
        tools=[search_summit_session],
        system_prompt="""あなたは、長期記憶と知識検索機能を備えたインテリジェントなイベントアシスタントです。
        search_summit_session を使用してセッション情報を検索してください。
        最初の検索で十分な情報が得られた場合は、追加のツール呼び出しを行わないでください。
        ユーザーの設定を記憶し、パーソナライズされたおすすめ情報を提供してください。
        """,
        state={
            "actor_id": actor_id,
            "session_id": session_id
        }
    )

# @app.entrypoit デコレータをつけた関数が
# Amazon Bedrock AgentCore Runtime から最初に呼ばれる
@app.entrypoint
def runtime_agent(payload, context):
    """
    AgentCore Runtime から呼び出されるメインのエントリーポイント
    """
    global agent
    
    user_input = payload.get("prompt")
    actor_id = payload.get("actor_id")
    session_id = context.session_id
    
    if not user_input:
        return "Error: Missing 'prompt' field"
    
    # 最初のリクエストが来たらエージェントを初期化
    if agent is None:
        initialize_agent(actor_id, session_id)
            
    # エージェントの呼び出し
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

## ステップ4.2: 実行ロールの作成

ランタイムが動作するために包括的なIAM権限が必要です。

> 💡 **ヘルパー関数**: 必要なすべての権限を持つロールを作成するためにユーティリティ関数を使用します。

> ⏱️ **伝播**: IAMロールの変更がグローバルに伝播するまで約30秒かかります。


In [ ]:
print("\n" + "=" * 70)
print("🔐 ランタイム用実行ロールの作成")
print("=" * 70)

agent_name = f"EventAgent{UNIQUE_ID}"
execution_role_arn = create_agentcore_role(agent_name=agent_name, region=REGION)

print(f"✅ 実行ポリシーをアタッチ完了")

print("\n⏳ ロールの伝播を待機中（30秒）...")
time.sleep(30)

## ステップ4.3: ランタイムの設定と起動

次に、デプロイを設定してBedrock AgentCoreランタイムに起動します。

### 設定パラメータ

- `entrypoint`: エージェントコードを含むPythonファイル（"event_agent.py"）
- `execution_role`: 権限のためのIAMロールARN
- `region`: デプロイ先のAWSリージョン
- `agent_name`: 一意識別子
- `auto_create_ecr`: コンテナレジストリを自動作成（True）
- `requirements_file`: Python依存関係（"requirements.txt"）
- `non_interactive`: ノートブックでのプロンプトを防止（True）
- `memory_mode`: "NO_MEMORY"（独自のメモリリソースを作成済み）

### 環境変数

環境変数経由で設定を注入します：
- `MEMORY_ID`: 使用するメモリリソース
- `KB_ID`: クエリするナレッジベース
- `MODEL_ID`: LLMモデル識別子
- `AWS_REGION`: AWSリージョン

> 💡 **監視**: ツールキットがリアルタイムの進捗更新を提供します。


In [ ]:
os.chdir("/home/sagemaker-user/amazon-bedrock-agentcore-samples/02-use-cases/intelligent-event-agent")      

In [ ]:
print("\n" + "=" * 70)
print("🚀 ランタイムの設定と起動")
print("=" * 70)

import subprocess
import json
import os
import shutil

# 作業ディレクトリを保存
original_dir = os.getcwd()

# ── 0. pyproject.toml を作成（CDK synthに必要） ──
if not os.path.exists("pyproject.toml"):
    with open("requirements.txt") as f:
        deps = [line.strip() for line in f if line.strip() and not line.startswith("#")]
    with open("pyproject.toml", "w") as f:
        f.write('[project]\nname = "event-agent"\nversion = "0.1.0"\nrequires-python = ">=3.10"\ndependencies = [\n')
        f.write(",\n".join(f'    "{d}"' for d in deps))
        f.write(",\n]\n")
    print("✅ pyproject.toml 作成完了")

# ── 1. AgentCore プロジェクト作成（エージェントなし） ──
print("\n📁 AgentCoreプロジェクトを初期化中...")

subprocess.run([
    "agentcore", "create",
    "--name", agent_name,
    "--no-agent",
    "--skip-git",
    "--skip-python-setup",
], check=True)
os.chdir(agent_name)

# ── 2. ソースコードを専用ディレクトリにコピー ──
# codeLocation="../" だと親ディレクトリ全体（.cacheを含む）をコピーし
# 再帰的にENAMETOOLONGエラーが発生するため、src/に分離する
os.makedirs("src", exist_ok=True)
for fname in ["event_agent.py", "requirements.txt", "utils.py", "pyproject.toml"]:
    src_path = os.path.join("..", fname)
    if os.path.exists(src_path):
        shutil.copy2(src_path, "src/")
print("✅ ソースコードを src/ にコピー完了")

# ── 3. 既存コードをBYOエージェントとして追加 ──
print(f"\n⚙️ エージェントを設定中: {agent_name}")

subprocess.run([
    "agentcore", "add", "agent",
    "--name", agent_name,
    "--type", "byo",
    "--framework", "Strands",
    "--code-location", "../",
    "--entrypoint", "event_agent.py",
    "--language", "Python",
    "--model-provider", "Bedrock",
], check=True)

# ── 4. デプロイターゲットを設定 ──
account_id = boto3.client("sts").get_caller_identity()["Account"]

with open("agentcore/aws-targets.json", "w") as f:
    json.dump([{"name": "default", "account": account_id, "region": REGION}], f, indent=2)
print(f"✅ デプロイターゲット設定完了 (account={account_id}, region={REGION})")

# ── 5. 環境変数と実行ロール、codeLocationを agentcore.json に反映 ──
config_path = "agentcore/agentcore.json"
with open(config_path) as f:
    config = json.load(f)

for runtime in config["runtimes"]:
    if runtime["name"] == agent_name:
        runtime["envVars"] = [
            {"name": "MEMORY_ID", "value": MEMORY_ID},
            {"name": "KB_ID", "value": KB_ID},
            {"name": "MODEL_ID", "value": MODEL_ID},
            {"name": "AWS_REGION", "value": REGION},
        ]
        runtime["executionRoleArn"] = execution_role_arn["Role"]["Arn"]
        # codeLocationを"src"に変更し、agentcore/.cacheの再帰コピーを防止
        runtime["codeLocation"] = "src"

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print("✅ 設定完了")

# ── 5.5. CDK schema version互換性エラー対策 ──
# agentcore CLI内蔵の@aws-cdk/cloud-assembly-schemaが古い場合に更新
print("📦 CDK互換性パッチ適用中...")
subprocess.run(
    ["npm", "install", "@aws-cdk/cloud-assembly-schema@latest"],
    cwd="/opt/conda/lib/node_modules/@aws/agentcore",
    check=True,
    capture_output=True,
)
print("✅ CDK互換性パッチ完了")

# ── 6. デプロイ ──
print("\n🚀 ランタイムをデプロイ中（約2〜3分かかります）...")

subprocess.run(["agentcore", "deploy", "-y"], check=True)
print("✅ デプロイ完了")

# ── 7. ステータス確認 ──
print("\n⏳ ランタイムのステータスを確認中...")

result = subprocess.run(
    ["agentcore", "status", "--runtime", agent_name, "--json"],
    capture_output=True, text=True,
)
print(result.stdout)

# 元のディレクトリに戻る
os.chdir(original_dir)
print("✅ ランタイム起動完了")


## ステップ4.4: ランタイムエージェントのテスト

デプロイされたエージェントが正しく動作していることをテストしましょう。

### セッション管理

ランタイムは自動的に：
- 提供されない場合は`session_id`を生成
- リクエスト間でセッションコンテキストを維持
- セッション同士を分離
- 会話の継続性を実現

> 💡 **同じactor_id**: 先ほど保存した好みにエージェントがアクセスできるよう、同じactor_idを使用します。


In [ ]:
print("\n" + "=" * 70)
print("🧪 ランタイムエージェントのテスト")
print("=" * 70)

# テスト用のセッションIDを作成
test_session_id = f"runtime-test-{str(uuid.uuid4())[:20]}"

# agentcore status コマンドの返却値から
# AgentCore Runtime の ARN を取得
import json, re
d = json.loads(re.sub(r'\x1b\[[^a-zA-Z]*[a-zA-Z]', '', result.stdout))   
runtime_arn = d['resources'][0]['identifier']

agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)

print(f"\n📝 ランタイムエージェントを呼び出し中...\n")
# actor_id を指定して AgentCore Runtime を呼び出す
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    runtimeSessionId=test_session_id,
    contentType="application/json",
    accept="application/json",
    payload=json.dumps({"prompt": "おすすめのセッションは何ですか?", "actor_id": ACTOR_ID}).encode(),
)

response_text = response["response"].read().decode("utf-8").strip('"').replace("\\n", "\n")
display(Markdown(response_text))

print("\n✅ ランタイムエージェントが動作しています！")
print("\n" + "=" * 70)
print("✅ 第4章完了: エージェントをランタイムにデプロイしました！")
print("=" * 70)


---

# 第5章: アイデンティティ

## 概要

この最終章では、AgentCoreアイデンティティとAmazon Cognitoを使用して**認証**を追加します。

### なぜAgentCoreアイデンティティなのか？

**本番環境の要件**:
- **認証**: ユーザーが誰であるかを検証
- **認可**: エージェントへのアクセスを制御
- **ユーザー分離**: 各ユーザーが独自のメモリ名前空間を取得
- **セキュリティ**: トークンベースの認証

<div style="text-align:left">
    <img src="architecture_images/image_10.png" width="80%"/>
</div>

### 実行する内容

1. Cognitoユーザープールを作成
2. パスワード付きのテストユーザーを作成
3. トークンを検証するようにエージェントコードを更新
4. 認証を含むランタイムを再設定
5. 認証済みエージェントをテスト


## ステップ5.1: Cognitoユーザープールの作成

Amazon Cognitoはユーザー管理と認証を提供します。

### Amazon Cognitoとは？

CognitoはAWSのマネージド認証サービスで、以下を提供します：
- **ユーザー管理**: ユーザーの作成、認証、管理
- **OAuth 2.0サポート**: 業界標準の認証
- **JWTトークン**: 安全でステートレスな認証トークン
- **MFAサポート**: 多要素認証（オプション）
- **ソーシャルログイン**: Google、Facebook、Apple（ここでは未使用）
- **SAML**: エンタープライズID連携

### JWTトークン構造

**アクセストークンのクレーム**:
```json
{
  "sub": "a1b2c3d4-e5f6-7890-abcd-ef1234567890",  // ユーザーID
  "token_use": "access",
  "scope": "openid profile",
  "auth_time": 1234567890,
  "iss": "https://cognito-idp.us-west-2.amazonaws.com/us-west-2_AbCdEfGhI",
  "exp": 1234571490,  // 1時間で有効期限切れ
  "iat": 1234567890,
  "client_id": "1234567890abcdefghijk",
  "username": "testuser"
}
```

### 作成されるもの

**ユーザープール**:
- ユーザーIDのコンテナ
- パスワードポリシー
- トークン設定

**アプリクライアント**:
- OAuth設定
- トークン設定
- 許可されたフロー

**テストユーザー**:
- ユーザー名: `testuser`
- パスワード: `TempPassword123!`
- ステータス: 確認済み

> 💡 **本番環境のヒント**: 本番環境では、既存のIDプロバイダーと統合するか、ユーザー登録フローを実装してください。


In [ ]:
print("\n" + "=" * 70)
print("🔐 Cognitoユーザープールの作成")
print("=" * 70)

from utils import setup_cognito_user_pool

print("\n🔐 Cognitoをセットアップ中...")
cognito_config = setup_cognito_user_pool(region=REGION)

print("\n✅ Cognitoセットアップ完了！")
print(f"   プールID: {cognito_config['pool_id']}")
print(f"   クライアントID: {cognito_config['client_id']}")
print(f"   ディスカバリURL: {cognito_config['discovery_url']}")
print(f"   テストユーザー: testuser / TempPassword123!")

## ステップ5.2: トークン検証を含むエージェントファイルの更新

次に、JWTトークンを検証する更新されたエージェントファイルを作成します。

### コードの変更点

**更新されたエントリポイント**:
```python
@app.entrypoint
def runtime_agent(payload, context):
    # ヘッダーからトークンを抽出
    auth_header = context.request_headers.get('Authorization')
    
    # 検証してユーザーIDを取得
    actor_id = get_user_sub(auth_header, REGION, COGNITO_USER_POOL)
    
    # actor_idをメモリ操作に使用
    ...
```

> ⚠️ **重要**: エージェントはパスワードを見ることはありません - 検証済みトークンのみです。


In [ ]:
%%writefile event_agent_with_auth.py
import os
import jwt
import json
import boto3
import logging
from strands import Agent, tool
from jwt import PyJWKClient
from typing import Dict, Any, List
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.memory.session import MemorySessionManager
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from bedrock_agentcore.memory.constants import StrategyType, ConversationalMessage, MessageRole


# logging の設定
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("event-agent")

# AgentCore App を初期化
app = BedrockAgentCoreApp()

# 各種環境変数を設定
MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
COGNITO_USER_POOL = os.getenv('COGNITO_USER_POOL')
REGION = os.getenv('AWS_REGION')
KB_ID = os.getenv('KB_ID')

agent = None
bedrock_agent_runtime = None

# Strands Agents が提供する HookProvider クラスを継承し
# MemoryHookProvider クラスを作成
class MemoryHookProvider(HookProvider):
    """Custom hook provider to integrate with AgentCore Memory"""
    
    def __init__(self):
        logger.info(f"Initializing MemoryHookProvider")
        self.memory_session_manager = MemorySessionManager(MEMORY_ID, REGION)
    
    # エージェント初期化時に長期メモリから User Preferences を検索
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load user preferences when agent starts"""
        logger.info("🎯 Agent initialization - loading preferences")
        
        # actor_id を取得        
        actor_id = event.agent.state.get("actor_id")
        
        if not actor_id:
            logger.warning("Missing actor_id")
            return
        
        try:
            # 長期メモリからユーザーの好みを取得
            preferences = self.memory_session_manager.search_long_term_memories(
                namespace_prefix=f"/users/{actor_id}/preferences",
                # 「ユーザーの設定・興味・背景」を問い合わせるクエリをハードコード
                query="What are the user's preferences, interests, and background?",
                # 類似情報を最大 5 件取得してメッセージに追加
                top_k=5
            )
            
            # User Preferences が取得できた場合
            if preferences:
                logger.info(f"✅ Loaded {len(preferences)} preferences")
                pref_messages = []
                # 取得した User Preferences のデータを整形
                for pref in preferences:
                    pref_text = pref.get('content', {}).get('text', '')
                    if pref_text:
                        try:
                            pref_json = json.loads(pref_text)
                            pref_messages.append(f"- {pref_json.get('preference', pref_text)}")
                        except:
                            pref_messages.append(f"- {pref_text}")
                
                if pref_messages:
                    context = "\n".join(pref_messages)

                    # User Preferences をメッセージに追加
                    # これによりエージェントはユーザーの好みに関するコンテキストを
                    # 持った状態で会話を開始できる
                    event.agent.system_prompt += f"\n\n**User Preferences:**\n{context}"
                    logger.info("✅ Added preferences to system prompt")
            else:
                logger.info("ℹ️ No preferences found yet")
                
        except Exception as e:
            logger.error(f"❌ Error loading preferences: {e}")
    
    # 新しいメッセージが追加される度に発火する
    def on_message_added(self, event: MessageAddedEvent):
        """Store messages in memory"""
        logger.info("💬 Message added - storing in memory")
        
        # actor_idを取得
        actor_id = event.agent.state.get("actor_id")
        # session_idを取得
        session_id = event.agent.state.get("session_id")
        
        if not all([actor_id, session_id]):
            logger.warning("Missing required state values")
            return
        
        try:
            messages = event.agent.messages
            # 最後のメッセージからコンテンツとロールを取得し、
            # AgentCore Memory のセッションマネージャーへ会話メッセージとして送信
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            if last_message["role"] == "user":
                message_role = MessageRole.USER
            elif last_message["role"] == "assistant":
                message_role = MessageRole.ASSISTANT
            
            # actor_id と session_id を指定して
            # 短期メモリにメッセージを送信
            # これにより論理的なデータ分離が実現
            self.memory_session_manager.add_turns(
                actor_id=actor_id,
                session_id=session_id,
                messages=[ConversationalMessage(message_content, message_role)]
            )
            logger.info("✅ Message stored")
            
        except Exception as e:
            logger.error(f"❌ Error storing message: {e}")
    
    # 作成した hooks を hook registory に登録
    # これによりエージェントは初期化時やメッセージ追加時に実行すべきコールバックを認識    
    def register_hooks(self, registry: HookRegistry):
        """Register hooks with the agent"""
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

# @tool デコレータをつけた関数はエージェントによってツールとして認識される
@tool
def search_summit_session(query: str) -> List[Dict[str, Any]]:
    # エージェントは docustring を読んで、ツールを実行するかどうかの判断を行う
    """
    セマンティック検索を使用して、ナレッジベースからAWS Summitのセッションを検索します。

    ユーザーがAWS Summitのセッションに関する推奨情報を求めた場合や、
    ユーザーの興味に基づいて関連するセッションを見つけたい場合に、このツールをご利用ください。

    Args:
        query: どのようなセッションを検索するかを指定する検索クエリ
        max_results: 返却するセッションの最大結果数 (default: 5, max: 10)

    Returns:
        関連するAWS Summit セッションの詳細が書かれたリスト
    """

    global bedrock_agent_runtime, KB_ID
    
    try:
        if not bedrock_agent_runtime:
            bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name=REGION)
        
        # ナレッジベース検索APIの呼び出し
        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={'text': query},
            retrievalConfiguration={
                'vectorSearchConfiguration': {
                    'numberOfResults': 5
                }
            }
        )
        
        # 結果の抽出とフォーマット
        results = []
        for idx, item in enumerate(response.get('retrievalResults', []), 1):
            results.append({
                'rank': idx,
                'content': item.get('content', {}).get('text', ''),
                'score': item.get('score', 0.0)
            })
        
        return results
    except Exception as e:
        return [{"error": str(e)}]

def get_user_sub(access_token: str, region: str, user_pool_id: str) -> str:
    """Cognito token からユーザー ID を抽出"""

    # 先頭7文字（Bearer ）をスライスで除去し、純粋なJWT文字列を取得
    access_token = access_token[7:]  # Remove 'Bearer ' prefix

    # JWKS(Json Web Key Set)の取得
    # Cognito は公開鍵を JWKS エンドポイントで公開している
    jwks_url = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/jwks.json"
    jwks_client = PyJWKClient(jwks_url)
    signing_key = jwks_client.get_signing_key_from_jwt(access_token)

    # トークンのデコードと検証を行う
    decoded = jwt.decode(
        access_token,
        signing_key.key,
        # RS256 で署名が正しいか確認
        algorithms=["RS256"],
        # iss クレームが指定した User Pool と一致するか検証
        issuer=f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}",
        # 以下で指定した必須クレームが全て存在するか確認
        options={"require": ["exp", "iat", "iss", "token_use"]}
    )

    # idトークンが誤って渡された場合を防ぐため
    # token_use が "access" であるかを明示的に確認
    if decoded.get("token_use") != "access":
        raise jwt.InvalidTokenError("Token is not an access token")

    # Cognito User Pool 内でユーザーを一意に識別する
    # UUID を返却
    return decoded["sub"]

def initialize_agent(actor_id, session_id):
    """エージェントの初期化"""
    global agent
    
    # Bedrock モデルの初期化
    model = BedrockModel(model_id=MODEL_ID)
    # MemoryHookProvider の初期化
    memory_hook = MemoryHookProvider()
    
    agent = Agent(
        model=model,
        hooks=[memory_hook],
        tools=[search_summit_session],
        system_prompt="""
        あなたは長期記憶を持つ、知的なイベントアシスタントです。
        search_summit_session を使用してセッション情報を検索してください。
        最初の検索で十分な情報が得られた場合は、追加のツール呼び出しを行わないでください。
        ユーザーの好みに基づいて、パーソナライズされたおすすめ情報を提供してください。
        """,
        state={
            "actor_id": actor_id,
            "session_id": session_id
        }
    )

# @app.entrypoit デコレータをつけた関数が
# Amazon Bedrock AgentCore Runtime から最初に呼ばれる
@app.entrypoint
def runtime_agent(payload, context):
    global agent
    
    # Cognito token から user ID(actor_id) を抽出
    auth_header = context.request_headers.get('Authorization')
    actor_id = get_user_sub(auth_header, REGION, COGNITO_USER_POOL)
    
    user_input = payload.get("prompt")
    session_id = context.session_id
    
    if not user_input:
        return "Error: Missing 'prompt' field"
    
    # 最初のリクエストが来たらエージェントを初期化
    if agent is None:
        initialize_agent(actor_id, session_id)
    
    # エージェントの呼び出し
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

## ステップ5.3: 認証を含むランタイムの再設定

次に、認証済みエージェントを使用するようにランタイムを再設定します。

### ランタイムオーソライザーの設定

**customJWTAuthorizer**:
```python
{
    "customJWTAuthorizer": {
        "discoveryUrl": "https://cognito-idp.region.amazonaws.com/poolId/.well-known/openid-configuration",
        "allowedClients": ["client-id-1", "client-id-2"]
    }
}
```

**これが行うこと**:
- ランタイムがJWTを自動的に検証
- 無効または期限切れのトークンを拒否
- 指定されたクライアントIDからのトークンのみを許可
- 有効なトークンをエージェントコードに転送

### リクエストヘッダーの設定

**requestHeaderAllowlist**:
```python
{"requestHeaderAllowlist": ["Authorization"]}
```

**これが重要な理由**:
- デフォルトでは、ランタイムはすべてのヘッダーをフィルタリング
- エージェントコードでAuthorizationヘッダーが必要
- 許可リストにより`context.request_headers`で利用可能に
- セキュリティ: 明示的に許可されたヘッダーのみが通過

### 更新された設定

**新しいパラメータ**:
- `entrypoint`: "event_agent_with_auth.py"（更新されたファイル）
- `request_header_configuration`: Authorizationヘッダーを転送
- `authorizer_configuration`: Cognito JWT検証

**新しい環境変数**:
- `COGNITO_USER_POOL`: トークン検証のためのプールID

> 💡 **二重セキュリティ**: ランタイムがトークンを検証し、さらにエージェントコードも再度検証する多層防御です。


In [ ]:
os.chdir("/home/sagemaker-user/amazon-bedrock-agentcore-samples/02-use-cases/intelligent-event-agent")   

In [ ]:
print("\n" + "=" * 70)
print("🔐 認証を含むランタイムの再設定")
print("=" * 70)

import subprocess
import json
import os
import shutil

auth_agent_name = f"auth_event_agent_{UNIQUE_ID}"
auth_project_name = f"AuthAgent{UNIQUE_ID}"  # 23文字以内
original_dir = os.getcwd()

# ── 0. pyproject.toml を作成（CDK synthに必要） ──
if not os.path.exists("pyproject.toml"):
    with open("requirements.txt") as f:
        deps = [line.strip() for line in f if line.strip() and not line.startswith("#")]
    with open("pyproject.toml", "w") as f:
        f.write('[project]\nname = "event-agent"\nversion = "0.1.0"\nrequires-python = ">=3.10"\ndependencies = [\n')
        f.write(",\n".join(f'    "{d}"' for d in deps))
        f.write(",\n]\n")
    print("✅ pyproject.toml 作成完了")

# ── 1. AgentCore プロジェクト作成 ──
print("\n📁 AgentCoreプロジェクトを初期化中...")

subprocess.run([
    "agentcore", "create",
    "--name", auth_project_name,
    "--no-agent",
    "--skip-git",
    "--skip-python-setup",
], check=True)
os.chdir(auth_project_name)

# ── 2. ソースコードを専用ディレクトリにコピー（再帰コピー防止） ──
os.makedirs("src", exist_ok=True)
for fname in ["event_agent_with_auth.py", "requirements.txt", "utils.py", "pyproject.toml"]:
    src_path = os.path.join("..", fname)
    if os.path.exists(src_path):
        shutil.copy2(src_path, "src/")
print("✅ ソースコードを src/ にコピー完了")

# ── 3. 認証付きBYOエージェントを追加 ──
print(f"\n⚙️ 認証付きエージェントを設定中: {auth_agent_name}")

subprocess.run([
    "agentcore", "add", "agent",
    "--name", auth_agent_name,
    "--type", "byo",
    "--framework", "Strands",
    "--code-location", "..",
    "--entrypoint", "event_agent_with_auth.py",
    "--language", "Python",
    "--model-provider", "Bedrock",
    "--authorizer-type", "CUSTOM_JWT",
    "--discovery-url", cognito_config["discovery_url"],
    "--allowed-clients", cognito_config["client_id"],
    "--request-header-allowlist", "Authorization",
], check=True)

# ── 4. デプロイターゲットを設定 ──
account_id = boto3.client("sts").get_caller_identity()["Account"]

with open("agentcore/aws-targets.json", "w") as f:
    json.dump([{"name": "default", "account": account_id, "region": REGION}], f, indent=2)
print(f"✅ デプロイターゲット設定完了 (account={account_id}, region={REGION})")

# ── 5. 環境変数と実行ロール、codeLocationを agentcore.json に反映 ──
config_path = "agentcore/agentcore.json"
with open(config_path) as f:
    config = json.load(f)

for runtime in config["runtimes"]:
    if runtime["name"] == auth_agent_name:
        runtime["envVars"] = [
            {"name": "MEMORY_ID", "value": MEMORY_ID},
            {"name": "COGNITO_USER_POOL", "value": cognito_config["pool_id"]},
            {"name": "KB_ID", "value": KB_ID},
            {"name": "MODEL_ID", "value": MODEL_ID},
            {"name": "AWS_REGION", "value": REGION},
        ]
        runtime["executionRoleArn"] = execution_role_arn["Role"]["Arn"]
        # codeLocationを"src"に変更し、agentcore/.cacheの再帰コピーを防止
        runtime["codeLocation"] = "src"

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print("✅ 設定完了")

# ── 5.5. CDK schema version互換性エラー対策 ──
print("📦 CDK互換性パッチ適用中...")
subprocess.run(
    ["npm", "install", "@aws-cdk/cloud-assembly-schema@latest"],
    cwd="/opt/conda/lib/node_modules/@aws/agentcore",
    check=True,
    capture_output=True,
)
print("✅ CDK互換性パッチ完了")

# ── 6. デプロイ ──
print("\n🚀 認証付きランタイムをデプロイ中（約2〜3分かかります）...")

subprocess.run(["agentcore", "deploy", "-y"], check=True)
print("✅ デプロイ完了")

# ── 7. ステータス確認 ──
print("\n⏳ ランタイムのステータスを確認中...")
result = subprocess.run(
    ["agentcore", "status", "--runtime", auth_agent_name, "--json"],
    capture_output=True, text=True,
)
print(result.stdout)

# deployed-state.jsonからRuntime ARNを取得
deployed_state_path = "agentcore/.cli/deployed-state.json"
if os.path.exists(deployed_state_path):
    with open(deployed_state_path) as f:
        state = json.load(f)
    for target in state.get("targets", {}).values():
        runtimes = target.get("resources", {}).get("runtimes", {})
        for name, info in runtimes.items():
            if auth_agent_name in name:
                auth_runtime_arn = info.get("runtimeArn", "")
                auth_runtime_id = info.get("runtimeId", "")
                print(f"✅ Runtime ARN: {auth_runtime_arn}")

os.chdir(original_dir)
print("✅ 認証付きランタイム起動完了")


## ステップ5.4: 認証済みエージェントのテスト

最後に、完全に認証されたエージェントをテストしましょう。


In [ ]:
print("\n" + "=" * 70)
print("🧪 認証済みエージェントのテスト")
print("=" * 70)

# Cognito からアクセストークンを取得する
cognito_client = boto3.client("cognito-idp", region_name=REGION)                                                                            
# ユーザー認証を開始
auth_result = cognito_client.initiate_auth(                                                                                                 
    ClientId=cognito_config["client_id"],                                                                                                   
    AuthFlow="USER_PASSWORD_AUTH",                                                                                                          
    AuthParameters={"USERNAME": "testuser", "PASSWORD": "MyPassword123!"},                                                                
)
# bearer_token を取得                                                                                                                                           
cognito_config["bearer_token"] = auth_result["AuthenticationResult"]["AccessToken"]   


import urllib.parse

# セッションIDを作成
auth_session_id = f"auth-test-{str(uuid.uuid4())[:25]}"
# ARN に含まれる ":", "/" をエンコード
escaped_arn = urllib.parse.quote(auth_runtime_arn, safe='')
url = f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{escaped_arn}/invocations?qualifier=DEFAULT"

# リクエストヘッダの設定
headers = {
    "Authorization": f"Bearer {cognito_config['bearer_token']}",
    "Content-Type": "application/json",
    "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": auth_session_id,
}

print(f"\n📝 認証付きで呼び出し中...\n")
# エージェントの呼び出し
response = requests.post(
    url,
    headers=headers,
    data=json.dumps({"prompt": "AIとセキュリティの観点から、どのようなセッションをおすすめしますか?"}),
)

if response.status_code == 200:
    # 結果を取得
    response_text = response.json()
    display(Markdown(f"**Agent Response:**\n\n{response_text}"))
else:
    print(f"❌ エラー ({response.status_code}): {response.text}")

print("\n✅ 認証済みエージェントが動作しています！")
print("  ユーザーはCognito経由で認証されました")
print("  ユーザーIDはトークンから抽出されました")
print("  メモリはユーザーごとに分離されています")

print("\n" + "=" * 70)
print("✅ 第5章完了: 認証が動作しています！")
print("=" * 70)


---

# 🎉 ノートブック完了！


## 最終アーキテクチャ

<div style="text-align:left">
    <img src="architecture_images/image_1.png" width="80%"/>
</div>

---

## 構築したもの

おめでとうございます！以下の機能を持つ完全な**高度なエージェント型RAGシステム**を構築しました：

### ✅ 第1章: エージェント
- Strandsベースのインテリジェントエージェント
- ツール統合フレームワーク
- 会話機能

### ✅ 第2章: ナレッジベース
- 583のre:Inventセッションドキュメントを含むS3バケット
- 効率的なベクトルストレージのためのS3 Vectors
- 自動取り込み機能付きBedrockナレッジベース
- セマンティック検索機能
- エージェント用カスタムRAGツール

### ✅ 第3章: メモリ
- 短期・長期ストレージを持つAgentCoreメモリ
- 自動学習のためのUserPreference戦略
- ライフサイクル制御のためのStrandsフック
- セッション横断メモリの永続化
- バックグラウンドでの好み抽出

### ✅ 第4章: ランタイム
- コンテナ化されたエージェントデプロイ
- スケーラビリティのためのAgentCoreランタイム
- 分離されたセッション実行
- 本番対応インフラ
- CloudWatchログとX-Rayトレース

### ✅ 第5章: アイデンティティ
- Amazon Cognito認証
- トークンベースの認可
- ユーザー分離
- 安全なアクセス制御
- JWT検証
---

## おめでとうございます！ 🎉

以下の構築とデプロイに成功しました：
- ✅ Strandsフレームワークによるインテリジェントエージェント
- ✅ ナレッジベースとS3 VectorsによるRAG
- ✅ 自動学習機能付き長期記憶
- ✅ AgentCoreランタイムへの本番デプロイ
- ✅ Cognitoによる安全な認証
- ✅ セッション横断メモリの永続化
- ✅ パーソナライズされた推奨



## リソースのクリーンアップ

継続的な課金を避けるため、リソースをクリーンアップすることを忘れないでください。

> ⚠️ **重要**: ワークショップが完全に終了した場合にのみ、これらのクリーンアップセルを実行してください。

In [ ]:
# すべてのリソースを削除する場合のみこのセルを実行してください
from utils import empty_and_delete_bucket

print("\n" + "=" * 70)
print("🗑️  クリーンアップ: リソースの削除")
print("=" * 70)

# 1. AgentCoreランタイムの削除
print("\n1. AgentCoreランタイムを削除中...")
try:
    agentcore_control_client = boto3.client(
        "bedrock-agentcore-control", region_name=REGION
    )
    

    # 最初のランタイムを削除
    runtime_delete_response = agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=status_response.config.agent_id,
    )
    print(f"   ✅ AgentCoreランタイム削除完了: {status_response.config.agent_id}")

    # 認証付きランタイムを削除
    runtime_delete_response = agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=status_response_auth.config.agent_id,
    )
    print(f"   ✅ AgentCoreランタイム削除完了: {status_response_auth.config.agent_id}")
except Exception as e:
    print(f"   ⚠️ AgentCoreランタイムの削除エラー: {e}")

# 2. ECRリポジトリの削除
print("\n2. ECRリポジトリを削除中...")
try:
    ecr_client = boto3.client("ecr", region_name=REGION)
    repository_name = status_response.config.ecr_repository.split("/")[1]

    response = ecr_client.delete_repository(
        repositoryName=repository_name,
        force=True,  # Force deletion even if it contains images
    )
    print(f"   ✅ ECRリポジトリ削除完了: {repository_name}")
except Exception as e:
    print(f"   ⚠️ ECRリポジトリの削除エラー: {e}")

# 3. メモリリソースの削除
print("\n3. メモリリソースを削除中...")
try:
    memory_manager.delete_memory(memory_id=MEMORY_ID)
    print(f"   ✅ メモリリソース削除完了: {MEMORY_ID}")
except Exception as e:
    print(f"   ⚠️ メモリリソースの削除エラー: {e}")

# 4. Cognitoユーザープールと関連リソースの削除
print("\n4. Cognitoユーザープールを削除中...")
try:
    cognito_client = boto3.client("cognito-idp", region_name=REGION)
    pool_id = cognito_config["pool_id"]

    # ユーザープールの削除
    cognito_client.delete_user_pool(UserPoolId=pool_id)
    print(f"   ✅ Cognitoユーザープール削除完了: {pool_id}")

except Exception as e:
    print(f"   ⚠️ Cognitoリソースの削除エラー: {e}")

# 5. IAMロールの削除
print("\n5. IAMロールを削除中...")


def delete_iam_role(role_arn):
    """IAMロールとそのすべてのポリシーを削除"""
    try:
        iam_client = boto3.client("iam", region_name=REGION)
        role_name = role_arn.split("/")[-1]

        # マネージドポリシーのデタッチ
        attached_policies = iam_client.list_attached_role_policies(RoleName=role_name)
        for policy in attached_policies.get("AttachedPolicies", []):
            iam_client.detach_role_policy(
                RoleName=role_name, PolicyArn=policy["PolicyArn"]
            )

        # インラインポリシーの削除
        inline_policies = iam_client.list_role_policies(RoleName=role_name)
        for policy_name in inline_policies.get("PolicyNames", []):
            iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)

        # ロールの削除
        iam_client.delete_role(RoleName=role_name)
        print(f"   ✅ IAMロール削除完了: {role_name}")
    except Exception as e:
        print(f"   ⚠️ IAMロールの削除エラー: {e}")


delete_iam_role(execution_role_arn["Role"]["Arn"])
delete_iam_role(kb_role_arn)

# 6. ナレッジベースの削除
print("\n6. ナレッジベースを削除中...")
try:
    bedrock_agent_client.delete_knowledge_base(knowledgeBaseId=KB_ID)
    print(f"   ✅ ナレッジベース削除完了: {KB_ID}")
except Exception as e:
    print(f"   ⚠️ ナレッジベースの削除エラー: {e}")

# 7. ベクトルストアの削除
print("\n7. ベクトルストアを削除中...")
try:
    s3vectors_client.delete_index(
        vectorBucketName=vector_store_name, indexName=vector_index_name
    )
    print(f"   ✅ S3ベクトルインデックス削除完了: {vector_index_name}")
except Exception as e:
    print(f"   ⚠️ ベクトルインデックスの削除エラー: {e}")

try:
    s3vectors_client.delete_vector_bucket_policy(vectorBucketName=vector_store_name)
    print(f"   ✅ S3ベクトルストアポリシー削除完了")
except Exception as e:
    print(f"   ⚠️ ベクトルストアポリシーの削除エラー: {e}")

try:
    s3vectors_client.delete_vector_bucket(vectorBucketName=vector_store_name)
    print(f"   ✅ S3ベクトルストア削除完了: {vector_store_name}")
except Exception as e:
    print(f"   ⚠️ ベクトルストアの削除エラー: {e}")

# 8. S3バケットの削除
print("\n8. S3バケットを削除中...")
try:
    empty_and_delete_bucket(BUCKET_NAME)
    print(f"   ✅ S3バケットを空にして削除完了: {BUCKET_NAME}")
except Exception as e:
    print(f"   ⚠️ S3バケットの削除エラー: {e}")

print("\n" + "=" * 70)
print("✅ クリーンアップ完了")
print("=" * 70)
print("\nすべてのリソースが削除されました。")